# Chapter 7 — Search In-Depth
## Topic 8: Metadata Filtering as a Search-Time Tool

**Run cells in order.**

- Module 1: Setup — knowledge base with metadata fields (doc_type, source)
- Module 2: Post-filter demonstration — the "fewer than k results" failure mode
- Module 3: Pre-filter (correct approach) — same query, correct behavior
- Module 4: Consistent filtering across hybrid retrieval (BM25 + Dense)
- Module 5: Soft filter (scoring boost) vs hard filter — head-to-head

**Install:** `pip install sentence-transformers rank-bm25 numpy`


## Module 1: Setup — Knowledge Base With Metadata

Deliberately skewed knowledge base: most chunks about "quarterly reporting"
are doc_type=faq, and only ONE genuinely relevant sop chunk exists —
designed so post-filtering can fail visibly.

**Requires:** nothing standalone (loads its own model)


In [ ]:
"""
MODULE 1: Setup

Knowledge base deliberately skewed so that a query's UNFILTERED top-k is
dominated by one doc_type, while the correct filtered doc_type match
exists but ranks lower in raw similarity -- this is what makes the
post-filter failure mode (Module 2) actually reproducible.
"""

import numpy as np
from sentence_transformers import SentenceTransformer

KNOWLEDGE_BASE = [
    {"id": 0, "text": "FAQ: What is the FD premature withdrawal penalty? It is 1 percent of the applicable rate.",
     "doc_type": "faq", "source": "01_FD_FAQ.pdf"},
    {"id": 1, "text": "FAQ: How is FD interest calculated? Interest is calculated quarterly on the principal.",
     "doc_type": "faq", "source": "01_FD_FAQ.pdf"},
    {"id": 2, "text": "FAQ: When is FD maturity interest credited? Within 2 working days of maturity.",
     "doc_type": "faq", "source": "01_FD_FAQ.pdf"},
    {"id": 3, "text": "FAQ: Can I renew my FD automatically? Yes, opt in via net banking before maturity.",
     "doc_type": "faq", "source": "01_FD_FAQ.pdf"},
    {"id": 4, "text": "SOP Step 1: Verify customer identity before processing any premature FD withdrawal penalty waiver request.",
     "doc_type": "sop", "source": "03_FD_SOPs.pdf"},
    {"id": 5, "text": "Policy: Senior citizens receive 0.5 percent additional interest on all FD tenures.",
     "doc_type": "policy", "source": "04_FD_Policy_Reference.pdf"},
    {"id": 6, "text": "Product Guide: 24-month FD offers 7.1 percent per annum interest rate.",
     "doc_type": "product", "source": "02_FD_Product_Guide.pdf"},
    {"id": 7, "text": "SOP Step 5: Complete the closure paperwork and update the account ledger accordingly.",
     "doc_type": "sop", "source": "03_FD_SOPs.pdf"},
]

CORPUS_TEXTS = [doc["text"] for doc in KNOWLEDGE_BASE]

print("Loading embedding model (may take a moment on first run)...")
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
kb_embeddings = model.encode(CORPUS_TEXTS, normalize_embeddings=True, show_progress_bar=False)
for doc in KNOWLEDGE_BASE:
    doc["embedding"] = kb_embeddings[doc["id"]]

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

print("\nKnowledge base:")
for doc in KNOWLEDGE_BASE:
    print(f"  Doc {doc['id']} [{doc['doc_type'].upper():<7}]: {doc['text'][:65]}...")

print("\nModule 1 complete. Run Module 2.")


## Module 2: Post-Filter — The "Fewer Than K" Failure Mode

Query specifically wants procedural (SOP) content, but there's only ONE
SOP chunk in the knowledge base. Post-filtering (search first, filter after)
against a small top-k can silently return zero results even though the
SOP chunk exists.

**Requires:** Module 1


In [ ]:
"""
MODULE 2: Post-Filter (search first, filter after) -- the anti-pattern

Demonstrates the exact failure mode flagged in the theory: if the
unfiltered top-K doesn't happen to include the target doc_type, post-
filtering returns fewer results than requested, or zero, even when
matching documents exist elsewhere in the corpus.
"""

QUERY = "how do I process a premature withdrawal penalty waiver"
REQUESTED_K = 3

def post_filter_search(query: str, doc_type: str, k: int, unfiltered_top_n: int = 3) -> list:
    """WRONG pattern: search top-N unfiltered, THEN discard non-matching doc_types."""
    query_vec = model.encode(query, normalize_embeddings=True, show_progress_bar=False)
    scored = [(doc, cosine_sim(query_vec, doc["embedding"])) for doc in KNOWLEDGE_BASE]
    scored.sort(key=lambda x: x[1], reverse=True)

    # Search only the top-N unfiltered results -- this is the bug:
    # real systems often set unfiltered_top_n = k, assuming filtering
    # won't remove much -- a dangerous assumption
    top_n = scored[:unfiltered_top_n]

    # NOW filter -- discard non-matching doc_types AFTER search
    filtered = [(doc, score) for doc, score in top_n if doc["doc_type"] == doc_type]
    return filtered[:k]


print(f"Query: '{QUERY}'")
print(f"Requested doc_type: sop, requested k={REQUESTED_K}\n")

result = post_filter_search(QUERY, doc_type="sop", k=REQUESTED_K, unfiltered_top_n=3)

print(f"Post-filter result: {len(result)} document(s) returned (requested {REQUESTED_K})")
for doc, score in result:
    print(f"  score={score:.4f} | Doc {doc['id']} [{doc['doc_type'].upper()}] {doc['text'][:60]}...")

if len(result) < REQUESTED_K:
    print(f"\n[FAILURE MODE REPRODUCED] Only {len(result)}/{REQUESTED_K} results returned.")
    print("Two SOP chunks (Doc 4 and Doc 7) genuinely exist in the corpus, but")
    print("post-filtering only searched the top-3 UNFILTERED results first --")
    print("Doc 7 was not similar enough to rank in that initial unfiltered top-3,")
    print("so it was never even considered, let alone filtered. Post-filtering")
    print("discards candidates it never properly searched for in the first place.")

print("\nModule 2 complete. Run Module 3.")


## Module 3: Pre-Filter — The Correct Approach

Same query, same doc_type constraint, but the filter is applied to the
FULL corpus BEFORE ranking, not after truncating to a small unfiltered
top-N. Guarantees the correct SOP chunk is found.

**Requires:** Module 1, Module 2


In [ ]:
"""
MODULE 3: Pre-Filter (filter during search, not after)

Correct pattern (mirrors Qdrant's query_filter applied during HNSW
traversal): restrict the candidate set to matching doc_type FIRST,
then rank ONLY within that matching subset.
"""

def pre_filter_search(query: str, doc_type: str, k: int) -> list:
    """CORRECT pattern: filter the corpus to matching doc_type FIRST,
    then rank only among the matches. Mirrors Qdrant's query_filter
    behavior during HNSW traversal (Chapter 6)."""
    query_vec = model.encode(query, normalize_embeddings=True, show_progress_bar=False)

    # Filter FIRST -- restrict the search space before any ranking happens
    matching_docs = [doc for doc in KNOWLEDGE_BASE if doc["doc_type"] == doc_type]

    if not matching_docs:
        return []

    # THEN rank only within the matching subset
    scored = [(doc, cosine_sim(query_vec, doc["embedding"])) for doc in matching_docs]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:k]


print(f"Query: '{QUERY}'")
print(f"Requested doc_type: sop, requested k={REQUESTED_K}\n")

result_prefilter = pre_filter_search(QUERY, doc_type="sop", k=REQUESTED_K)

print(f"Pre-filter result: {len(result_prefilter)} document(s) returned (requested {REQUESTED_K})")
for doc, score in result_prefilter:
    print(f"  score={score:.4f} | Doc {doc['id']} [{doc['doc_type'].upper()}] {doc['text'][:60]}...")

print("\n" + "=" * 65)
print("POST-FILTER vs PRE-FILTER -- DIRECT COMPARISON")
print("=" * 65)
print(f"Post-filter returned: {len(result)} results")
print(f"Pre-filter returned:  {len(result_prefilter)} results")
print("\nPre-filtering guarantees: if k matching documents exist ANYWHERE")
print("in the corpus, they will be found -- the filter narrows the SEARCH")
print("SPACE, not the RESULT SET after the fact.")

print("\nModule 3 complete. Run Module 4.")


## Module 4: Consistent Filtering Across Hybrid Retrieval (BM25 + Dense)

Applies the SAME doc_type filter to both BM25 and dense retrieval before
RRF fusion (Topic 4) — demonstrates the correctness/security bug that
results from filtering only one retriever.

**Requires:** Module 1


In [ ]:
"""
MODULE 4: Consistent Filtering Across BM25 + Dense (Hybrid, Topic 4)

Shows why filtering must be applied to BOTH retrievers before RRF fusion --
filtering only one creates a correctness bug, and if the filter exists for
access-control reasons (Chapter 6 Topic 9), a genuine PII leak.
"""

from rank_bm25 import BM25Okapi

def tokenize(text: str) -> list:
    return text.lower().split()


def bm25_filtered_search(query: str, doc_type: str, k: int = 10) -> list:
    """BM25 restricted to matching doc_type BEFORE scoring -- consistent
    with the dense pre-filter pattern from Module 3."""
    matching_docs = [doc for doc in KNOWLEDGE_BASE if doc["doc_type"] == doc_type]
    if not matching_docs:
        return []
    tokenized = [tokenize(doc["text"]) for doc in matching_docs]
    bm25 = BM25Okapi(tokenized, k1=1.2, b=0.75)
    scores = bm25.get_scores(tokenize(query))
    scored = [(matching_docs[i]["id"], scores[i]) for i in range(len(scores)) if scores[i] > 0]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:k]


def bm25_unfiltered_search(query: str, k: int = 10) -> list:
    """BM25 over the FULL corpus, no filter -- the inconsistency bug."""
    tokenized = [tokenize(doc["text"]) for doc in KNOWLEDGE_BASE]
    bm25 = BM25Okapi(tokenized, k1=1.2, b=0.75)
    scores = bm25.get_scores(tokenize(query))
    scored = [(KNOWLEDGE_BASE[i]["id"], scores[i]) for i in range(len(scores)) if scores[i] > 0]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:k]


query = "premature withdrawal penalty"

print("=" * 65)
print("CORRECT: Same doc_type filter applied to BOTH retrievers")
print("=" * 65)

dense_filtered = [(doc["id"], score) for doc, score in pre_filter_search(query, doc_type="sop", k=10)]
bm25_filt = bm25_filtered_search(query, doc_type="sop", k=10)

print(f"\nDense (filtered to sop):  {dense_filtered}")
print(f"BM25 (filtered to sop):   {bm25_filt}")
print("Both retrievers ONLY ever see SOP documents -- fusion cannot")
print("surface a non-SOP document under any circumstance. Consistent.")

print("\n" + "=" * 65)
print("BUG: Filter applied to dense only, BM25 left unfiltered")
print("=" * 65)

bm25_unfilt = bm25_unfiltered_search(query, k=10)
bm25_unfilt_doc_types = [next(d for d in KNOWLEDGE_BASE if d["id"] == doc_id)["doc_type"]
                          for doc_id, _ in bm25_unfilt]

print(f"\nDense (filtered to sop):     {dense_filtered}")
print(f"BM25 (UNFILTERED, all types): {bm25_unfilt}")
print(f"BM25 result doc_types:        {bm25_unfilt_doc_types}")

non_sop_leaked = [dt for dt in bm25_unfilt_doc_types if dt != "sop"]
if non_sop_leaked:
    print(f"\n[BUG REPRODUCED] Non-SOP doc_types leaked through BM25's")
    print(f"unfiltered path: {set(non_sop_leaked)}")
    print("If fused via RRF (Topic 4), these non-SOP documents CAN surface")
    print("in the final ranking despite the filter -- exactly the correctness")
    print("bug (and, for access-control filters, PII leak) flagged in the theory.")

print("\nModule 4 complete. Run Module 5.")


## Module 5: Soft Filter (Scoring Boost) vs Hard Filter

Compares hard pre-filtering (exclude non-matching entirely) against a
soft filter (boost matching documents' scores without excluding others)
— shows the recall-preservation benefit of the soft approach when the
filter signal might be wrong.

**Requires:** Module 1


In [ ]:
"""
MODULE 5: Soft Filter vs Hard Filter

Soft filter: boost matching doc_type's score instead of excluding
non-matching documents entirely -- preserves recall if the upstream
filter signal turns out to be wrong for a specific query.
"""

def soft_filter_search(query: str, preferred_doc_type: str, boost: float = 0.15, k: int = 3) -> list:
    """Boosts matching doc_type's score by a fixed amount, but does NOT
    exclude other doc_types -- they remain eligible."""
    query_vec = model.encode(query, normalize_embeddings=True, show_progress_bar=False)
    scored = []
    for doc in KNOWLEDGE_BASE:
        base_score = cosine_sim(query_vec, doc["embedding"])
        boosted_score = base_score + boost if doc["doc_type"] == preferred_doc_type else base_score
        scored.append((doc, boosted_score, base_score))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:k]


# Scenario: the upstream classification signal suggests "sop" but is WRONG --
# the query is actually better answered by an FAQ entry
query_ambiguous = "what happens to my FD interest at maturity"

print(f"Query: '{query_ambiguous}'")
print("(Upstream signal suggests doc_type=faq, but let's test what happens")
print(" if that signal is applied as HARD vs SOFT toward doc_type=sop instead,")
print(" simulating a case where the upstream signal guessed wrong)\n")

# Hard filter to (wrongly guessed) sop
hard_result = pre_filter_search(query_ambiguous, doc_type="sop", k=3)
print("HARD filter to doc_type=sop (wrong guess):")
if hard_result:
    for doc, score in hard_result:
        print(f"  score={score:.4f} | Doc {doc['id']} [{doc['doc_type'].upper()}] {doc['text'][:55]}...")
else:
    print("  (no results -- the only SOP doc is unrelated to this query)")

# Soft filter (boost) toward sop -- but faq/other docs remain eligible
soft_result = soft_filter_search(query_ambiguous, preferred_doc_type="sop", boost=0.15, k=3)
print("\nSOFT filter (boost) toward doc_type=sop (wrong guess), others still eligible:")
for doc, boosted, base in soft_result:
    marker = " (boosted)" if doc["doc_type"] == "sop" else ""
    print(f"  score={boosted:.4f} (base={base:.4f}){marker} | Doc {doc['id']} [{doc['doc_type'].upper()}] {doc['text'][:50]}...")

print("""
Key difference demonstrated:
  When the upstream filter signal is WRONG, hard filtering can return
  nothing useful at all (or the wrong document, forced through). Soft
  filtering still finds the genuinely correct FAQ answer, because
  non-matching doc_types remain eligible -- the boost only helps SOP
  content win close contests, it doesn't exclude everything else.

Trade-off: soft filtering gives up some of hard filtering's precision
  when the signal IS correct, in exchange for resilience when it's wrong.
  Which is better depends on how confident the upstream signal actually
  is -- validate with Topic 9's evaluation harness, not assumption.
""")

print("=" * 70)
print("CHAPTER 7 TOPIC 8 SUMMARY")
print("=" * 70)
print("""
Post-filter (search first, filter after): can silently return fewer
  than k results, or zero, if the unfiltered top-N doesn't happen to
  contain matches -- demonstrated directly in Module 2.

Pre-filter (filter during search): guarantees matching documents are
  found if they exist anywhere in the corpus -- Qdrant's query_filter
  during HNSW traversal implements this natively (Chapter 6).

Hybrid retrieval (Topic 4) requires the SAME filter applied to BOTH
  BM25 and dense retrieval before fusion -- filtering only one is both
  a correctness bug and, for access-control filters, a PII leak risk
  (Module 4 reproduced this directly).

Hard filter vs soft filter (boost): hard filtering maximizes precision
  when the filter signal is correct but can eliminate the right answer
  entirely when it's wrong; soft filtering preserves recall at some
  precision cost. Choice should be validated against Topic 9's
  evaluation harness, not decided by default.

Next: Topic 9 -- Evaluating Retrieval (Recall@K, MRR, NDCG@K)
""")
